# Document AI and Structured Extraction Pipelines

**Phase 10** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-10/02-document-ai.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic pdfplumber pydantic pymupdf

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

A legal team receives 50 contracts per day as PDFs. Junior associates read each one and copy the key fields into a spreadsheet: party names, effective date, termination clause, governing law. It takes 15 minutes per contract. The team wants to automate it.

The engineering team starts exploring. They immediately hit four questions:

1. Should we use OCR and then send the text to Claude, or send each PDF page as an image directly?
2. How do we handle a 40-page contract? We cannot send 40 images in one API call within budget.
3. What accuracy can we realistically promise the legal team? If the system misses a termination date in a contract, there are real consequences.
4. Some contracts are sc...

## The Concept

### Two paths for document AI

The core decision is whether to extract text first (cheaper, faster) or treat the page as an image (required for scanned documents, better for form layouts).



**Text extraction path (born-digital PDFs):**
- Use `pdfplumber` or `PyMuPDF` to extract the raw text from each page
- Send the extracted text to Claude as a standard text prompt
- Cost: ~2,000-5,000 tokens per contract page vs ~37,000 tokens for the same page as an image at 768px
- Accuracy: ~97% for standard contract fields when the text is cleanly extracted

**Vision path (scanned or form-heavy PDFs):*...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 10-02: Document AI and Structured Extraction Pipelines
Detects PDF type (born-digital vs scanned), routes to appropriate extraction path,
and returns validated structured data using Pydantic.

Usage:
    python main.py                  # demo mode (uses embedded contract text)
    python main.py contract.pdf     # process a real PDF file

Optional dependencies:
    pip install pymupdf pydantic    # for PDF parsing
    pip install pdfplumber pydantic # alternative PDF parser
"""

import anthropic
import base64
import json
import sys
from pathlib import Path
from typing import Optional

try:
    from pydantic import BaseModel, Field
except ImportError:
    raise SystemExit("Install pydantic: pip install pydantic")


# --------------------------------------------------------------------------- #
# Output schema                                                                #
# --------------------------------------------------------------------------- #

class ContractExtraction(BaseModel):
    party_a: Optional[str] = Field(None, description="First party name")
    party_b: Optional[str] = Field(None, description="Second party name")
    effective_date: Optional[str] = Field(None, description="Contract effective date")
    termination_clause: Optional[str] = Field(None, description="Summary of termination conditions")
    governing_law: Optional[str] = Field(None, description="Governing law jurisdiction")
    confidence: str = Field("low", description="high / medium / low")
    extraction_notes: list[str] = Field(default_factory=list)


# --------------------------------------------------------------------------- #
# PDF type detection                                                           #
# --------------------------------------------------------------------------- #

def is_born_digital(pdf_bytes: bytes, min_text_chars: int = 100) -> bool:
    """
    Returns True if the PDF has enough embedded text for text extraction.
    Falls back to a byte-level heuristic if no PDF library is available.
    """
    try:
        import fitz
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        total_text = ""
        for page in doc:
            total_text += page.get_text()
            if len(total_text) >= min_text_chars:
                doc.close()
                return True
        doc.close()
        return False
    except ImportError:
        pass

    try:
        import pdfplumber
        import io
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            total = "".join((p.extract_text() or "") for p in pdf.pages)
            return len(total) >= min_text_chars
    except ImportError:
        pass

    # Heuristic: born-digital PDFs reference /Font objects
    return b"/Font" in pdf_bytes


def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    """Extract all text from a born-digital PDF."""
    try:
        import fitz
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        pages = [page.get_text() for page in doc]
        doc.close()
        return "\n\n--- PAGE BREAK ---\n\n".join(pages)
    except ImportError:
        pass

    try:
        import pdfplumber
        import io
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            pages = [p.extract_text() or "" for p in pdf.pages]
        return "\n\n--- PAGE BREAK ---\n\n".join(pages)
    except ImportError:
        pass

    raise RuntimeError(
        "No PDF library available. Install PyMuPDF: pip install pymupdf"
    )


def pdf_pages_to_images(pdf_bytes: bytes, dpi: int = 150) -> list[bytes]:
    """Convert PDF pages to PNG images (requires PyMuPDF)."""
    try:
        import fitz
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        images = []
        for page in doc:
            pix = page.get_pixmap(matrix=mat)
            images.append(pix.tobytes("png"))
        doc.close()
        return images
    except ImportError:
        raise RuntimeError("Install PyMuPDF: pip install pymupdf")


# --------------------------------------------------------------------------- #
# Extraction prompt                                                            #
# --------------------------------------------------------------------------- #

EXTRACTION_PROMPT = """Extract the following fields from this contract document.
Return valid JSON only, no markdown fences, no explanation.

JSON structure:
{
  "party_a": "First party name or null",
  "party_b": "Second party name or null",
  "effective_date": "Date string or null",
  "termination_clause": "Brief summary of termination conditions or null",
  "governing_law": "Jurisdiction or null",
  "confidence": "high or medium or low",
  "extraction_notes": ["any caveats"]
}

Confidence levels:
- high: all five main fields found clearly
- medium: 3-4 fields found, or one is ambiguous
- low: fewer than 3 fields found"""


# --------------------------------------------------------------------------- #
# Extraction functions                                                         #
# --------------------------------------------------------------------------- #

def extract_from_text(
    text: str, model: str = "claude-3-5-haiku-20241022"
) -> ContractExtraction:
    """Text extraction path for born-digital PDFs."""
    client = anthropic.Anthropic()

    # Truncate to stay within context limits
    if len(text) > 150_000:
        text = text[:150_000] + "\n\n[DOCUMENT TRUNCATED]"

    message = client.messages.create(
        model=model,
        max_tokens=512,
        messages=[
            {"role": "user", "content": f"{EXTRACTION_PROMPT}\n\nDOCUMENT TEXT:\n{text}"}
        ],
    )
    raw = message.content[0].text.strip().lstrip("```json").rstrip("```").strip()
    return ContractExtraction(**json.loads(raw))


def extract_from_images(
    page_images: list[bytes],
    model: str = "claude-3-5-haiku-20241022",
    max_pages: int = 5,
) -> ContractExtraction:
    """Vision extraction path for scanned PDFs."""
    client = anthropic.Anthropic()

    content: list[dict] = []
    for i, img_bytes in enumerate(page_images[:max_pages]):
        b64 = base64.standard_b64encode(img_bytes).decode("utf-8")
        content.append({
            "type": "image",
            "source": {"type": "base64", "media_type": "image/png", "data": b64},
        })
        content.append({
            "type": "text",
            "text": f"[Page {i + 1} of {len(page_images)}]",
        })

    content.append({"type": "text", "text": EXTRACTION_PROMPT})

    message = client.messages.create(
        model=model,
        max_tokens=512,
        messages=[{"role": "user", "content": content}],
    )
    raw = message.content[0].text.strip().lstrip("```json").rstrip("```").strip()
    return ContractExtraction(**json.loads(raw))


# --------------------------------------------------------------------------- #
# Demo data                                                                    #
# --------------------------------------------------------------------------- #

DEMO_CONTRACT_TEXT = """
SERVICE AGREEMENT

This Service Agreement ("Agreement") is entered into as of January 15, 2025 ("Effective Date")
between Acme Corporation, a Delaware corporation ("Party A"), and BuildRight LLC,
a California limited liability company ("Party B").

1. SERVICES. Party B agrees to provide software development services as described in Exhibit A.

2. TERM AND TERMINATION. This Agreement commences on the Effective Date and continues for
twelve (12) months. Either party may terminate this Agreement upon thirty (30) days written notice.
Party A may terminate immediately for cause if Party B materially breaches this Agreement.

3. GOVERNING LAW. This Agreement shall be governed by the laws of the State of Delaware,
without regard to its conflict of law provisions.

4. PAYMENT. Party A shall pay Party B $15,000 per month within 30 days of invoice.

5. SIGNATURES. Executed as of the date first written above.
"""

# --------------------------------------------------------------------------- #
# Main                                                                         #
# --------------------------------------------------------------------------- #

def main():
    print("=== Lesson 10-02: Document AI and Structured Extraction Pipelines ===\n")

    if len(sys.argv) > 1:
        pdf_path = Path(sys.argv[1])
        if not pdf_path.exists():
            print(f"Error: file not found: {pdf_path}")
            sys.exit(1)

        print(f"Processing: {pdf_path}")
        pdf_bytes = pdf_path.read_bytes()
        print(f"File size: {len(pdf_bytes):,} bytes")

        print("\nDetecting PDF type...")
        born_digital = is_born_digital(pdf_bytes)
        print(f"  Born-digital: {born_digital}")
        print(f"  Path: {'text extraction' if born_digital else 'vision (images)'}")

        if born_digital:
            text = extract_text_from_pdf(pdf_bytes)
            print(f"  Extracted text: {len(text):,} characters")
            print("\nSending to Claude (text path)...")
            result = extract_from_text(text)
        else:
            pages = pdf_pages_to_images(pdf_bytes)
            print(f"  Pages converted to images: {len(pages)}")
            print("\nSending to Claude (vision path, first 5 pages)...")
            result = extract_from_images(pages)
    else:
        print("No PDF provided. Running demo with embedded contract text.")
        print("(Text extraction path)\n")
        result = extract_from_text(DEMO_CONTRACT_TEXT)

    print("\n--- Extraction Result ---")
    print(json.dumps(result.model_dump(), indent=2))

    print("\n--- Routing Decision ---")
    if result.confidence == "high":
        print("  -> WRITE TO DATABASE: all fields extracted with high confidence")
    elif result.confidence == "medium":
        print("  -> HUMAN REVIEW: medium confidence, verify uncertain fields")
    else:
        print("  -> HUMAN REVIEW QUEUE: low confidence, full document review needed")

    if result.extraction_notes:
        print("\n--- Notes ---")
        for note in result.extraction_notes:
            print(f"  * {note}")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the primary business reason to implement born-digital vs scanned detection?**

- A. The vision path fails entirely for born-digital PDFs
- B. The text extraction path is 10-20x cheaper per page and typically produces better accuracy for born-digital PDFs
- C. Vision models cannot process born-digital PDFs due to PDF rendering issues
- D. Text extraction is faster but less accurate than vision for all document types

**2. Which chunking strategy is best suited for extracting specific contract fields (parties, dates, governing law) from a 40-page born-digital contract?**

- A. Process pages one at a time and stop when all required fields are populated
- B. Always send only page 1 since all contract fields appear on the first page
- C. Split the document into 10-page batches and send each batch separately
- D. Use summarization to compress each page before extraction

**3. Why is per-field accuracy tracking more useful than overall document accuracy?**

- A. Per-field tracking is required by GDPR for legal document processing
- B. Different fields have different difficulty profiles; tracking them separately reveals which fields need prompt improvements or human review routing
- C. Overall accuracy cannot be computed without per-field accuracy
- D. Per-field tracking reduces the number of documents needed in the golden set

**4. What does this finding indicate, and what should the team do?**

- A. The Pydantic validation is broken; fix the schema
- B. The confidence label in the prompt is not well-calibrated; revise the prompt to make 'high' criteria more strict
- C. 30% error rate is acceptable for AI extraction; no action needed
- D. Switch to a larger model to improve confidence accuracy

**5. What design change would reduce review time most effectively?**

- A. Show the reviewer only the extracted JSON and the original PDF, side by side
- B. Include page number and surrounding text snippet for each extracted field so reviewers can jump directly to the relevant passage
- C. Increase the confidence threshold so fewer documents go to review
- D. Add a second LLM call to double-check all extracted fields before routing

**6. For this use case, which library choice matters most and why?**

- A. PyMuPDF is strictly required because pdfplumber cannot handle multi-page PDFs
- B. pdfplumber is strictly required because PyMuPDF does not support text extraction
- C. Both work for text extraction; the choice depends on license requirements and how the library handles complex layouts like tables
- D. Neither works; only Docling can extract text from contracts reliably

**7. In what specific scenario does Docling add clear value over pdfplumber for structured extraction?**

- A. Docling adds value for any PDF type since it uses a larger AI model
- B. Docling preserves table structure as markdown tables, which improves extraction of cell-aligned data like payment schedules and SLA tables
- C. Docling is required for all scanned documents since pdfplumber cannot handle images
- D. Docling reduces token costs compared to pdfplumber because its output is shorter

_Answers are in `checks.json` in the lesson directory._